# Full Demo: Options Pricing & Greeks Calculator

End-to-end demonstration of the options-pricer library.
Fetches real market data for AAPL, prices options, computes Greeks, and visualizes results.

In [ ]:
# Setup — imports, load .env, fetch MarketData for AAPL
import os
from dotenv import load_dotenv
load_dotenv()

from src.data import get_market_data, get_rfr
from src.models import bsm_price, binomial_price, mc_price
from src.greeks import all_greeks
from src.volatility import build_surface_df, VolSurface
from src.plots import (payoff_diagram, greeks_profile, mc_paths_chart,
                       vol_surface_3d, vol_smile_chart)
import pandas as pd

print("Fetching AAPL market data...")
md = get_market_data("AAPL")
print(f"Ticker: {md.ticker}")
print(f"Spot: ${md.spot:.2f}")
print(f"Risk-free rate: {md.rfr:.2%}")
print(f"Historical vol (30d): {md.hist_vol:.2%}")
print(f"Option chain expiries: {list(md.option_chain.keys()) if md.option_chain else 'None'}")

## Market Snapshot

In [ ]:
# Market snapshot — print spot, RFR, hist vol; show last 30 days price chart
import matplotlib.pyplot as plt

print(f"Spot: ${md.spot:.2f}")
print(f"Risk-free rate: {md.rfr:.2%}")
print(f"Historical vol (30d): {md.hist_vol:.2%}")
print(f"Data fetched at: {md.fetched_at}")

# Last 30 days price chart
hist = md.price_history.tail(30)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hist.index, hist['Close'], color='#58a6ff', lw=2)
ax.set_title(f"AAPL Last 30 Trading Days — Spot ${md.spot:.2f}")
ax.set_ylabel("Price ($)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Price One Option

In [ ]:
# Price one option — AAPL nearest ATM call and put, all three models, comparison table
if md.option_chain:
    # Get nearest expiry
    expiry = list(md.option_chain.keys())[0]
    calls = md.option_chain[expiry]['calls']
    puts  = md.option_chain[expiry]['puts']
    
    # Find ATM strike (closest to spot)
    atm_strike = calls.iloc[(calls['strike'] - md.spot).abs().argsort()[:1]]['strike'].values[0]
    
    # Time to expiry
    from datetime import datetime, date
    exp_date = datetime.strptime(expiry, "%Y-%m-%d").date()
    T = max((exp_date - date.today()).days, 1) / 365.0
    
    print(f"Expiry: {expiry} (T={T:.4f} yr)")
    print(f"ATM Strike: {atm_strike}")
    print(f"Spot: {md.spot:.2f}, RFR: {md.rfr:.4f}, σ: {md.hist_vol:.4f}")
    
    # Price with all three models
    bsm_call  = bsm_price(md.spot, atm_strike, T, md.rfr, md.hist_vol, "call")
    bin_call  = binomial_price(md.spot, atm_strike, T, md.rfr, md.hist_vol, "call", steps=500)
    mc_call   = mc_price(md.spot, atm_strike, T, md.rfr, md.hist_vol, "call", n_sims=100_000)
    
    bsm_put   = bsm_price(md.spot, atm_strike, T, md.rfr, md.hist_vol, "put")
    bin_put   = binomial_price(md.spot, atm_strike, T, md.rfr, md.hist_vol, "put", steps=500)
    mc_put    = mc_price(md.spot, atm_strike, T, md.rfr, md.hist_vol, "put", n_sims=100_000)
    
    # Comparison table
    comp = pd.DataFrame({
        "Model": ["BSM", "Binomial(500)", "MC(100k)"],
        "Call Price": [bsm_call.price, bin_call.price, mc_call.price],
        "Put Price":  [bsm_put.price, bin_put.price, mc_put.price],
    })
    print(comp.to_string(index=False))
    print(f"\nMC Call 95% CI: [{mc_call.meta['ci_lower']:.4f}, {mc_call.meta['ci_upper']:.4f}]")
    print(f"MC Put 95% CI:  [{mc_put.meta['ci_lower']:.4f}, {mc_put.meta['ci_upper']:.4f}]")
else:
    print("No option chain available for AAPL")

## Greeks Table

In [ ]:
# Greeks table — all_greeks() for the same option, print formatted
if md.option_chain:
    g_call = all_greeks(md.spot, atm_strike, T, md.rfr, md.hist_vol, "call")
    g_put  = all_greeks(md.spot, atm_strike, T, md.rfr, md.hist_vol, "put")
    
    greeks_df = pd.DataFrame({
        "Greek": ["Delta", "Gamma", "Theta ($/day)", "Vega ($/1% σ)", 
                  "Rho ($/1% r)", "Vanna", "Volga", "Charm (Δ/day)"],
        "Call": [g_call.delta, g_call.gamma, g_call.theta, g_call.vega,
                 g_call.rho, g_call.vanna, g_call.volga, g_call.charm],
        "Put":  [g_put.delta, g_put.gamma, g_put.theta, g_put.vega,
                 g_put.rho, g_put.vanna, g_put.volga, g_put.charm],
    })
    print(greeks_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
else:
    print("No option chain available for AAPL")

## Payoff Diagram

In [ ]:
# Payoff diagram — call plots.payoff_diagram()
if md.option_chain:
    fig = payoff_diagram(md.spot, atm_strike, bsm_call.price, "call")
    print(f"Saved to outputs/payoff_call.png")
    # Display in notebook
    import matplotlib.pyplot as plt
    plt.show()

## Greeks Profile

In [ ]:
# Greeks profile — call plots.greeks_profile()
if md.option_chain:
    fig = greeks_profile(atm_strike, T, md.rfr, md.hist_vol)
    print(f"Saved to outputs/greeks_profile.png")
    import matplotlib.pyplot as plt
    plt.show()

## MC Paths

In [ ]:
# MC paths — simulate and call plots.mc_paths_chart()
if md.option_chain:
    from src.models import mc_simulate_paths
    paths = mc_simulate_paths(md.spot, T, md.rfr, md.hist_vol, n_paths=500, n_steps=252, seed=42)
    fig = mc_paths_chart(paths, atm_strike, T, "call")
    print(f"Saved to outputs/mc_paths.png")
    import matplotlib.pyplot as plt
    plt.show()

## Vol Surface

In [ ]:
# Vol surface — build_surface_df() → VolSurface().fit() → plots.vol_surface_3d()
if md.option_chain:
    surface_df = build_surface_df(md.option_chain, md.spot, md.rfr)
    print(f"Surface data points: {len(surface_df)}")
    if len(surface_df) >= 16:
        surface = VolSurface().fit(surface_df)
        fig = vol_surface_3d(surface_df, "AAPL")
        print(f"Saved to outputs/vol_surface_AAPL.html")
        # In notebook, display the interactive plot
        fig.show()
    else:
        print("Insufficient data points for surface fit (need ≥ 16)")

## Vol Smile

In [ ]:
# Vol smile — plots.vol_smile_chart() for 4 expiries
if md.option_chain and len(surface_df) >= 16:
    # Get 4 expiries with data
    expiries_with_data = sorted(surface_df['T'].unique())[:4]
    print(f"Plotting smiles for T = {expiries_with_data}")
    fig = vol_smile_chart(surface, md.spot, expiries_with_data)
    print(f"Saved to outputs/vol_smile.png")
    import matplotlib.pyplot as plt
    plt.show()
else:
    print("Cannot plot smiles — no surface data or insufficient points")